In [7]:
# conda vis2
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [8]:
from src.ExperimentSet import ExperimentSet

In [9]:
es_dir = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V08'
exset = ExperimentSet(es_dir)
exset.set_retrieval_list()
exset.load_retrievals(iteration=-1, extinction_th=[0.3, np.inf])

In [24]:
for i in range(len(list(exset.retrievals.keys()))):
    print(f'{i}: {list(exset.retrievals.keys())[i]}')

0: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/CVM/LCL_CTH
1: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/CVM/unconstrained
2: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/CVM/unconstrained_darksurf
3: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/CVM/unconstrained_mod
4: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/HBC/periodic
5: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/aerosol/0p150
6: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/aerosol/0p250
7: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/aerosol/mix22
8: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/aerosol/mix54
9: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/cloud_motion/fast
10: /data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V08/cloud_motion/fast_unconstrained
11: /data/gdi/b/johntl3/LundstromPub2025/e

In [26]:
vmin = -10
vmax = 10

# cloud = r.solvers_retrieval[0.67].medium['cloud']
cmpr = list(exset.retrievals.values())[1].solvers_retrieval[0.67].medium['cloud']
ref = list(exset.retrievals.values())[13].solvers_retrieval[0.67].medium['cloud']

X, Y, Z = np.meshgrid(cmpr.x.data, cmpr.y.data, cmpr.z.data, indexing='ij')
data = go.Volume(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=(cmpr.extinction - ref.extinction).data.flatten(),
    isomin=vmin,
    isomax=vmax, 
    opacity=0.1,
    surface_count=33,
    showscale=True,
    colorbar=dict(
        title='Difference',
        titleside='right',
        tickmode='array',
        tickvals=[vmin, 0.5*(vmin+vmax), vmax],
        ticktext=['Low', 'Medium', 'High'],
        len=0.8,
        thickness=20
    )
)
layout = go.Layout(autosize=False, width=750, height=650)
fig = go.Figure(data=data, layout=layout)
pad = {'b':0, 'l':0, 'r':0, 't':0, 'pad':0}
font = {'size':16}
scene = dict(
    xaxis=dict(title='x (km)', tickvals=[0, 4.4, 8.8, 13.2, 17.6], range=[-1, 18.6]),
    yaxis=dict(title='y (km)', tickvals=[0, 4.4, 8.8, 13.2, 17.6], range=[-1, 18.6]),
    zaxis=dict(title='z (km)', tickvals=[0, 1.4], range=[-0.1, 1.5]),
)

theta = np.radians(45)
phi = -0.3*np.pi
radius = 2.5
h = radius * np.sin(theta)

camera = dict(
    up=dict(x=0, y=0, z=1.0),
    center=dict(x=0.0, y=0.0, z=0.0),
    eye=dict(x=h*np.cos(phi), y=h*np.sin(phi), z=radius*np.cos(theta))
)
fig.update_layout(
    scene_camera=camera,
    margin=pad,
    font=font,
    scene=scene
)